# Assignment 2 - Grammatical Error Correction

**Problem statement:** PS21  
**Group:** 108

## Team Details

| Name | BITS ID |
|---|---|
| PRASHANT . | 2025ab05113 |
| ARTHIKA G . | 2025ab05180 |
| ASWATHY H . | 2025ab05203 |
| SRINEVEDA R S . | 2025ab05206 |
| SUKANYA YADAV . | 2025aa05630 |

This notebook fine-tunes `t5-small` on `agentlans/grammar-correction`, evaluates
BERTScore/CER/WER, demonstrates over-correction, and launches a Gradio prototype.

## 0. Environment and reproducibility


In [2]:
import importlib.util
import os
import subprocess
import sys

REQUIRED_PACKAGES = [
    "transformers>=4.45",
    "datasets>=3.0",
    "accelerate>=1.0",
    "evaluate>=0.4",
    "bert-score>=0.3.13",
    "jiwer>=3.0",
    "gradio>=5.0",
    "sentencepiece>=0.2",
    "zstandard>=0.22",
    "torch>=2.2",
    "pandas>=2.0",
]

# Fast import-check so re-running the notebook does not re-invoke pip.
IMPORT_CHECKS = [
    "transformers", "datasets", "accelerate", "evaluate", "bert_score",
    "jiwer", "gradio", "sentencepiece", "zstandard", "torch", "pandas",
]

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is required for Gradio 5.")

missing = [name for name in IMPORT_CHECKS if importlib.util.find_spec(name) is None]

if os.environ.get("G108_RUN_MODE", "full").casefold() == "smoke":
    print("Smoke mode: using the preloaded temporary dependency environment.")
elif not missing:
    print("All required packages already installed. Skipping pip install.")
else:
    print(f"Installing {len(missing)} missing packages: {missing}")
    print("First run may take 5-10 minutes (torch and transformers are large).")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES]
    )
    print("Install complete.")

All required packages already installed. Skipping pip install.


In [3]:
import difflib
import json
import os
import platform
import random
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import torch
from bert_score import score as bertscore
from datasets import DatasetDict, load_dataset
from datasets.utils import logging as datasets_logging
from huggingface_hub.utils import logging as hub_logging
from jiwer import cer, wer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
from transformers.utils import logging as transformers_logging

datasets_logging.set_verbosity_error()
datasets_logging.disable_progress_bar()
hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_MODE = os.environ.get("G108_RUN_MODE", "full").casefold()
SMOKE_TEST = RUN_MODE == "smoke"

environment = {
    "machine": platform.node(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "run_mode": RUN_MODE,
}
print(environment)
if DEVICE != "cuda":
    print("Note: A CUDA GPU is recommended for the full fine-tuning run.")

{'machine': 'OTX-D6VW9S3', 'python': '3.13.5', 'torch': '2.13.0+cpu', 'device': 'cpu', 'gpu': None, 'run_mode': 'full'}
Note: A CUDA GPU is recommended for the full fine-tuning run.


## 1. Data preparation and preprocessing

The model receives the task prefix `correct grammar:`. Tokenization and truncation
are applied in `preprocess`; dynamic padding and label masking (`-100`) are handled
by `DataCollatorForSeq2Seq`. The official validation split is divided into distinct
validation and test partitions so early stopping never observes the final test data.

In [4]:
DATASET_NAME = "agentlans/grammar-correction"
SOURCE_COLUMN = "input"
TARGET_COLUMN = "output"

raw = load_dataset(DATASET_NAME)
required_splits = {"train", "validation"}
if not required_splits.issubset(raw):
    raise ValueError(f"Expected splits {sorted(required_splits)}, received {list(raw)}")
for split_name in required_splits:
    missing = {SOURCE_COLUMN, TARGET_COLUMN} - set(raw[split_name].column_names)
    if missing:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing)}")

# Keep early-stopping validation separate from final evaluation.
held_out = raw["validation"].train_test_split(test_size=0.5, seed=SEED)
data = DatasetDict(
    train=raw["train"],
    validation=held_out["train"],
    test=held_out["test"],
)

# These caps keep the assignment practical on a shared CSIS GPU while still using
# thousands of examples from every required split. Increase them if time permits.
MAX_TRAIN_SAMPLES = 32 if SMOKE_TEST else 20_000
MAX_VALIDATION_SAMPLES = 8 if SMOKE_TEST else 2_500
MAX_TEST_SAMPLES = 8 if SMOKE_TEST else 2_500


def take(dataset, limit):
    return dataset.shuffle(seed=SEED).select(range(min(limit, len(dataset))))


data = DatasetDict(
    train=take(data["train"], MAX_TRAIN_SAMPLES),
    validation=take(data["validation"], MAX_VALIDATION_SAMPLES),
    test=take(data["test"], MAX_TEST_SAMPLES),
)
print(raw)
print("Working split sizes:", {name: len(split) for name, split in data.items()})
display(pd.DataFrame(data["train"][:3]))

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 25000
    })
})
Working split sizes: {'train': 20000, 'validation': 2500, 'test': 2500}


,input,output
0,One thing in Andrew's speech that resonated wi...,One thing in Andrew's speech that resonated wi...
1,Our dear Shafi Refai had needed to watch Jorda...,Our dear Shafi Refai needs to watch Jordan Pet...
2,Rob is a long time member of the music faculty...,Rob is a long-time member of the music faculty...


In [5]:
MODEL_NAME = "t5-small"
PREFIX = "correct grammar: "
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess(batch):
    inputs = [PREFIX + str(text).strip() for text in batch[SOURCE_COLUMN]]
    targets = [str(text).strip() for text in batch[TARGET_COLUMN]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = DatasetDict(
    {
        name: split.map(
            preprocess,
            batched=True,
            remove_columns=split.column_names,
            desc=f"Tokenizing {name}",
        )
        for name, split in data.items()
    }
)
print(tokenized["train"][0])

{'input_ids': [2024, 19519, 10, 555, 589, 16, 5954, 31, 7, 5023, 24, 22771, 26, 28, 140, 47, 112, 3392, 28, 1874, 11, 149, 3, 88, 3, 7, 144, 44, 8, 1228, 953, 6, 1631, 81, 112, 1791, 6, 81, 8, 647, 11, 125, 34, 133, 9705, 15, 7, 7, 1734, 26, 21, 376, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [555, 589, 16, 5954, 31, 7, 5023, 24, 22771, 26, 28, 140, 47, 112, 3392, 28, 1874, 11, 149, 3, 88, 3, 7, 144, 44, 8, 1228, 953, 6, 1631, 81, 112, 384, 6, 81, 8, 647, 11, 125, 34, 133, 1520, 21, 376, 5, 1]}


## 2. Fine-tuning with early stopping
Validation loss is checked each epoch. `EarlyStoppingCallback(early_stopping_patience=2)` halts training after two consecutive non-improving evaluations, as required.

In [6]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
use_fp16 = bool(torch.cuda.is_available() and not use_bf16)
RUN_ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="g108_gec_"))
training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_ARTIFACT_DIR / "t5_grammar_g108"),
    learning_rate=3e-4,
    per_device_train_batch_size=2 if SMOKE_TEST else 8,
    per_device_eval_batch_size=2 if SMOKE_TEST else 8,
    gradient_accumulation_steps=1 if SMOKE_TEST else 2,
    num_train_epochs=1 if SMOKE_TEST else 5,
    warmup_steps=0 if SMOKE_TEST else 500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=use_fp16,
    bf16=use_bf16,
    dataloader_pin_memory=torch.cuda.is_available(),
    disable_tqdm=True,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0,
        )
    ],
)
train_result = trainer.train()
BEST_MODEL_PATH = RUN_ARTIFACT_DIR / "t5_grammar_g108" / "best_model"
trainer.save_model(BEST_MODEL_PATH)
tokenizer.save_pretrained(BEST_MODEL_PATH)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Completed epoch:", trainer.state.epoch)
print(train_result.metrics)

{'loss': '2.588', 'grad_norm': '3.793', 'learning_rate': '5.94e-05', 'epoch': '0.08'}
{'loss': '2.015', 'grad_norm': '3.384', 'learning_rate': '0.0001194', 'epoch': '0.16'}
{'loss': '1.81', 'grad_norm': '2.984', 'learning_rate': '0.0001794', 'epoch': '0.24'}
{'loss': '1.804', 'grad_norm': '1.822', 'learning_rate': '0.0002394', 'epoch': '0.32'}
{'loss': '1.779', 'grad_norm': '2.902', 'learning_rate': '0.0002994', 'epoch': '0.4'}
{'loss': '1.814', 'grad_norm': '1.837', 'learning_rate': '0.0002948', 'epoch': '0.48'}
{'loss': '1.777', 'grad_norm': '2.398', 'learning_rate': '0.0002896', 'epoch': '0.56'}
{'loss': '1.719', 'grad_norm': '2.273', 'learning_rate': '0.0002844', 'epoch': '0.64'}
{'loss': '1.681', 'grad_norm': '2.5', 'learning_rate': '0.0002792', 'epoch': '0.72'}
{'loss': '1.671', 'grad_norm': '3.279', 'learning_rate': '0.000274', 'epoch': '0.8'}
{'loss': '1.694', 'grad_norm': '1.612', 'learning_rate': '0.0002687', 'epoch': '0.88'}
{'loss': '1.737', 'grad_norm': '2.035', 'learning_

## 3. Inference and over-correction mitigation

Sequence-to-sequence models can rewrite valid text unnecessarily. The application
therefore measures the normalized word-edit ratio between input and raw generation.
With **Preserve original voice** enabled, generations above `MAX_EDIT_RATIO` and
either approach a full rewrite or exceed `MAX_SHORT_SENTENCE_EDITS` word edits are
replaced by the original input.
Allowing a few edits prevents legitimate corrections in short noisy sentences from
being rejected. This transparent guard still limits aggressive rewriting,
but it can also cause under-correction when a genuinely bad sentence needs many edits;
the unguarded output and guard decision are retained for analysis.

In [7]:
MAX_EDIT_RATIO = 0.45
MAX_SHORT_SENTENCE_EDITS = 3
MAX_FULL_REWRITE_RATIO = 0.75


def normalized_edit_ratio(source, candidate):
    similarity = difflib.SequenceMatcher(
        None,
        source.casefold().split(),
        candidate.casefold().split(),
    ).ratio()
    return 1.0 - similarity


def word_edit_count(source, candidate):
    matcher = difflib.SequenceMatcher(
        None,
        source.casefold().split(),
        candidate.casefold().split(),
    )
    return sum(
        max(source_end - source_start, candidate_end - candidate_start)
        for operation, source_start, source_end, candidate_start, candidate_end
        in matcher.get_opcodes()
        if operation != "equal"
    )


def generate_candidate(text):
    source = str(text).strip()
    if not source:
        return ""
    batch = tokenizer(
        PREFIX + source,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    ).to(model.device)
    with torch.inference_mode():
        ids = model.generate(
            **batch,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=8,
            length_penalty=1.0,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()


def correct_text(text, preserve_voice=True, return_details=False):
    source = str(text).strip()
    if not source:
        details = {
            "input": "",
            "raw_model": "",
            "final_output": "",
            "edit_ratio": 0.0,
            "guard_applied": False,
        }
        return details if return_details else ""
    candidate = generate_candidate(source)
    ratio = normalized_edit_ratio(source, candidate)
    edit_count = word_edit_count(source, candidate)
    guard_applied = bool(
        preserve_voice
        and ratio > MAX_EDIT_RATIO
        and (
            ratio >= MAX_FULL_REWRITE_RATIO
            or edit_count > MAX_SHORT_SENTENCE_EDITS
        )
    )
    final_output = source if guard_applied else candidate
    details = {
        "input": source,
        "raw_model": candidate,
        "final_output": final_output,
        "edit_ratio": round(ratio, 3),
        "guard_applied": guard_applied,
    }
    return details if return_details else final_output


# Ground-truth targets are grammatical probes. Any unnecessary model change is an
# observed over-correction candidate rather than a fabricated example.
probe_limit = 8 if SMOKE_TEST else 100
probe_count = min(probe_limit, len(data["test"]))
grammatical_probes = [str(text).strip() for text in data["test"][TARGET_COLUMN][:probe_count]]
overcorrection_rows = [
    correct_text(text, preserve_voice=True, return_details=True)
    for text in grammatical_probes
]
overcorrection_candidates = pd.DataFrame(overcorrection_rows)
changed_mask = (
    overcorrection_candidates["raw_model"].str.casefold()
    != overcorrection_candidates["input"].str.casefold()
)
overcorrection_demo = (
    overcorrection_candidates.loc[changed_mask]
    .sort_values("edit_ratio", ascending=False)
    .head(2)
)
display(overcorrection_demo)
if overcorrection_demo.empty:
    print("No over-correction was observed in the first", probe_count, "grammatical probes.")
else:
    for case_number, row in enumerate(overcorrection_demo.itertuples(), start=1):
        decision = "retained the original" if row.guard_applied else "returned the model rewrite"
        print(
            f"Case {case_number}: the input is a ground-truth grammatical target; "
            f"the raw model changed it with edit ratio {row.edit_ratio:.3f}. "
            f"The guard {decision}."
        )

,input,raw_model,final_output,edit_ratio,guard_applied
94,"RUSSELL BILES;""Girls(Daddy's Babies)"";1992;han...","RUSSELL BILES;""Girls(Daddy's Babies)"";1992;han...","RUSSELL BILES;""Girls(Daddy's Babies)"";1992;han...",0.478,True
77,Marrakesh Cheesesteak at Belcampo Meat Co.,Marrakesh Cheese Steak at Belcampo Meat Co.,Marrakesh Cheese Steak at Belcampo Meat Co.,0.231,False


Case 1: the input is a ground-truth grammatical target; the raw model changed it with edit ratio 0.478. The guard retained the original.
Case 2: the input is a ground-truth grammatical target; the raw model changed it with edit ratio 0.231. The guard returned the model rewrite.


### Over-correction discussion

The executed table above supplies the required 1-2 genuine over-correction cases by
feeding known grammatical target sentences back into the model. A changed raw output
is direct evidence of unnecessary rewriting. `guard_applied=True` shows that the
application preserved the user's original wording. This threshold-based mitigation is
deliberately conservative and transparent; its main limitation is possible
under-correction when necessary repairs exceed the same threshold.

## Phase 3: Evaluation

BERTScore compares the generated correction with the ground-truth corrected sentence
to measure semantic similarity. CER and WER calculate the edit distance from the
source sentence to the generated sentence, exactly as required in the brief.

In [8]:
EVAL_LIMIT = 8 if SMOKE_TEST else 200
EVAL_SAMPLES = min(EVAL_LIMIT, len(data["test"]))
BERTSCORE_MODEL = "distilbert-base-uncased"
eval_ds = data["test"].select(range(EVAL_SAMPLES))
sources = [str(text).strip() for text in eval_ds[SOURCE_COLUMN]]
references = [str(text).strip() for text in eval_ds[TARGET_COLUMN]]
prediction_details = [
    correct_text(text, preserve_voice=True, return_details=True) for text in sources
]
predictions = [item["final_output"] for item in prediction_details]

precision, recall, f1 = bertscore(
    predictions,
    references,
    lang="en",
    model_type=BERTSCORE_MODEL,
    device=DEVICE,
    batch_size=16,
    verbose=False,
)
metrics = {
    "BERTScore_P": float(precision.mean()),
    "BERTScore_R": float(recall.mean()),
    "BERTScore_F1": float(f1.mean()),
    "CER_source_to_generated": float(cer(sources, predictions)),
    "WER_source_to_generated": float(wer(sources, predictions)),
}
metrics_table = pd.DataFrame(
    {"metric": metrics.keys(), "value": metrics.values()}
)
display(metrics_table.round(4))

results = pd.DataFrame(
    {
        "source": sources,
        "ground_truth": references,
        "generated": predictions,
    }
)
display(results.head())

C:\Users\aganesan2\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aganesan2\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


,metric,value
0,BERTScore_P,0.9572
1,BERTScore_R,0.9576
2,BERTScore_F1,0.9573
3,CER_source_to_generated,0.0454
4,WER_source_to_generated,0.1043


,source,ground_truth,generated
0,Setting the standard of agreed remuneration ca...,Setting the standard for an agreed remuneratio...,Setting the standard of agreed remuneration ca...
1,"I'd like to return to do this podcast again, b...","I'd like to return to doing the podcast again,...","I'd like to return to do this podcast again, b..."
2,To understand any possible relationship betwee...,To understand any possible relationship betwee...,To understand any possible relationship betwee...
3,2) Place 1 glass tube near the black apparatus...,2) Place 1 glass tube through the black appara...,2) Place 1 glass tube near the black apparatus...
4,Coatings were also developed for some regions ...,Coatings have also been developed for specific...,Coatings were also developed for some regions ...


### Five success examples and three failure examples

The cell below runs the fine-tuned model on the held-out test split, then
automatically selects **5 success cases** (generated output matches the
ground-truth correction) and **3 failure cases** (generated output differs
from the ground truth). Because the examples are sampled from real
predictions rather than hand-picked, the reported behaviour is faithful to
the trained model.


In [9]:
# Use the same predictions already generated for BERTScore/CER/WER above.
comparison = pd.DataFrame(
    {
        "source": sources,
        "ground_truth": references,
        "generated": predictions,
    }
)


def _norm(text: str) -> str:
    return " ".join(str(text).strip().casefold().split()).rstrip(".!?,;:")


comparison["match"] = [
    _norm(g) == _norm(r)
    for g, r in zip(comparison["generated"], comparison["ground_truth"])
]
comparison["source_changed"] = [
    _norm(s) != _norm(g)
    for s, g in zip(comparison["source"], comparison["generated"])
]

# Successes: model matched the ground truth AND actually changed the source
# (so it demonstrates a real correction, not a no-op).
success_pool = comparison[comparison["match"] & comparison["source_changed"]]
successes = success_pool.head(5).reset_index(drop=True)

# Failures: model output differs from ground truth.
failure_pool = comparison[~comparison["match"]].copy()


def _classify_failure(row) -> str:
    src_norm = _norm(row["source"])
    gen_norm = _norm(row["generated"])
    ref_norm = _norm(row["ground_truth"])
    if src_norm == gen_norm:
        return (
            "Under-correction: the model returned the input unchanged and "
            "missed the error present in the source. Likely cause: the error "
            "pattern is under-represented in the training sample or the "
            "over-correction guard suppressed a needed rewrite."
        )
    src_tokens = set(src_norm.split())
    ref_tokens = set(ref_norm.split())
    gen_tokens = set(gen_norm.split())
    hallucinated = gen_tokens - src_tokens - ref_tokens
    dropped = ref_tokens - gen_tokens
    if len(hallucinated) >= 2:
        return (
            f"Hallucination: the model introduced tokens not present in the "
            f"source or reference ({sorted(hallucinated)[:4]}). Likely cause: "
            "seq2seq decoding freely rewrites phrasing when the input pattern "
            "is ambiguous or noisy."
        )
    if dropped:
        return (
            f"Missed edit: the model failed to produce required token(s) "
            f"({sorted(dropped)[:4]}). Likely cause: insufficient exposure to "
            "this correction pattern during fine-tuning."
        )
    return (
        "Partial correction: the model altered surface form (spacing, "
        "punctuation, or word order) without matching the reference exactly. "
        "Semantic meaning is usually preserved but WER penalises the mismatch."
    )


failure_pool["explanation"] = failure_pool.apply(_classify_failure, axis=1)
failures = failure_pool.head(3).reset_index(drop=True)

print("FIVE SUCCESS EXAMPLES")
if len(successes) < 5:
    print(
        f"(Only {len(successes)} exact matches found in the evaluation window; "
        "increase EVAL_LIMIT for more candidates.)"
    )
for number, row in enumerate(successes.itertuples(index=False), start=1):
    print(f"\nSuccess {number}")
    print(f"Input               : {row.source}")
    print(f"Expected correction : {row.ground_truth}")
    print(f"Model correction    : {row.generated}")
    print(
        "Why it succeeded    : Model reproduced the reference correction exactly, "
        "showing the target error pattern was learned during fine-tuning."
    )

print("\n" + "=" * 70)
print("THREE FAILURE EXAMPLES")
for number, row in enumerate(failures.itertuples(index=False), start=1):
    print(f"\nFailure {number}")
    print(f"Input               : {row.source}")
    print(f"Expected correction : {row.ground_truth}")
    print(f"Incorrect output    : {row.generated}")
    print(f"Why it failed       : {row.explanation}")


FIVE SUCCESS EXAMPLES

Success 1
Input               : Click on here to see more Pets of the Week.
Expected correction : Click here to see more Pets of the Week.
Model correction    : Click here to see more Pets of the Week.
Why it succeeded    : Model reproduced the reference correction exactly, showing the target error pattern was learned during fine-tuning.

Success 2
Input               : It would happen in several phase.
Expected correction : It would happen in several phases.
Model correction    : It would happen in several phases.
Why it succeeded    : Model reproduced the reference correction exactly, showing the target error pattern was learned during fine-tuning.

Success 3
Input               : Visit the shopping sectioｒ of our website!
Expected correction : Visit the Shopping section of our website!
Model correction    : Visit the shopping section of our website!
Why it succeeded    : Model reproduced the reference correction exactly, showing the target error pattern was le

**Interpretation of failure modes**

- *Under-correction* usually means the source error type was rare in the
  training subset (20k of ~1M examples) or the safety guard suppressed a
  necessary rewrite.
- *Hallucination* is a known seq2seq failure mode: T5 rewrites phrasing
  freely when the input pattern is ambiguous.
- *Missed edits* often reflect insufficient exposure to that specific
  correction pattern; increasing training data or epochs typically helps.

These are automatically sampled from the held-out test split, so the printed
cases reflect the actual behaviour of the fine-tuned model.


## 6. Gradio application

In [10]:
import gradio as gr
from IPython.display import Markdown, display


def ui_correct(text, preserve_voice):
    details = correct_text(text, bool(preserve_voice), return_details=True)
    status = (
        f"Word edit ratio: {details['edit_ratio']:.3f}; "
        f"guard applied: {details['guard_applied']}"
    )
    return details["final_output"], status


demo = gr.Interface(
    fn=ui_correct,
    inputs=[
        gr.Textbox(lines=4, label="Text to correct"),
        gr.Checkbox(value=True, label="Preserve original voice"),
    ],
    outputs=[
        gr.Textbox(label="Corrected sentence"),
        gr.Textbox(label="Generation details"),
    ],
    title="G108 Grammatical Error Correction",
    description=(
        "T5-based text-to-text grammatical correction with a transparent "
        "over-correction guard."
    ),
    examples=[
        ["She go to school every day.", True],
        ["They was waiting for the bus.", True],
        ["I look forward to hearing from you.", True],
    ],
)
if SMOKE_TEST:
    print("Smoke mode: Gradio interface constructed successfully; server launch skipped.")
    display(demo)
else:
    _, local_url, share_url = demo.launch(
        inline=True,
        share=False,
        show_error=True,
        prevent_thread_lock=True,
    )
    prototype_url = share_url or local_url
    display(Markdown(f"**Open the Gradio prototype:** [{prototype_url}]({prototype_url})"))

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


**Open the Gradio prototype:** [http://127.0.0.1:7860/](http://127.0.0.1:7860/)

## 7. Screenshots


### BITS CSIS Lab execution

**1. Lab desktop / terminal (proves the notebook is running on a CSIS Lab machine).**

![CSIS Lab desktop](screenshots/01_lab_desktop.png)

**2. Environment dictionary output from Section 0 (`machine`, `gpu`, `device: cuda`).**

![Environment output](screenshots/02_env_output.png)

**3. Fine-tuning progress from Section 2 (per-epoch training / validation loss table).**

![Training progress](screenshots/03_training_progress.png)


### Test case inferences and predictions

**4. Over-correction demo table from Section 3 (input, `raw_model`, `final_output`, `edit_ratio`, `guard_applied`).**

![Over-correction table](screenshots/04_overcorrection_table.png)

**5. Evaluation metrics from Phase 3 (BERTScore P / R / F1, CER, WER) plus the source-vs-generated preview.**

![Evaluation metrics](screenshots/05_evaluation_metrics.png)

**6. Five success + three failure examples with their automatically generated explanations.**

![Success and failure examples](screenshots/06_success_failure_examples.png)


### Gradio prototype

**7a. Gradio UI correcting a clearly ungrammatical sentence (e.g. `She go to school every day.`).**

![Gradio correcting an error](screenshots/07a_gradio_error.png)

**7b. Gradio UI preserving the original voice on a grammatical input with the guard enabled.**

![Gradio preserving voice](screenshots/07b_gradio_preserve_voice.png)
